In [ ]:
%load_ext autoreload
%autoreload 2

import sys

from starhoppingpy import zfuncs as zf  # Import as a regular module
#from starhopping.sh_funcs import irdis_makeflat, irdis_make_sky, sph_cen_irdis, \
#    irdis_make_flux_cube, irdis_basicreduc, irdis_make_asdi_cube, sph_remove_bad_images

from starhoppingpy.sh_funcs import irdis_makeflat, irdis_make_sky, sph_cen_irdis, \
    irdis_make_flux_cube, irdis_basicreduc, irdis_make_asdi_cube, sph_remove_bad_images

import os


##P109data
DATA_DIR = "/path/to/SORT/dayYYYY-MM-DD-cidNNNNNNN-IRDIS"  # set to your sorted data folder
os.chdir(DATA_DIR)

In [ ]:
irdis_makeflat('flatlist.txt')

In [ ]:
irdis_make_sky('skylist.txt')
if not os.path.exists('sky_irdis.fits'):
    irdis_badpix_from_obj2('scilist.txt')


In [ ]:
sph_cen_irdis('centerlist.txt',lmaskoutw0=30, rmaskoutw0=30)

In [ ]:
#For Testing bscentrd
import importlib
from starhoppingpy import basic
importlib.reload(basic)  # Reloads the module

from starhoppingpy import zfuncs as zf  # Import as a regular module

from starhoppingpy.basic import bscentrd
img = zf.readfits('temp.fits')
bscentrd(img,516,486)


In [ ]:
##DO THE FLUX CALCULATION LATER##
##irdis_make_flux_cube('fluxlist.txt',szn=501)

In [ ]:
irdis_basicreduc('scilist.txt')


In [ ]:
##CHECKING IF THERE IS A MISMATCH B/W HEADER NDIT AND ACTUAL CUBE SHAPE, AND REMOVING IF ANY
##FOR THE TIME BEING. 

import numpy as np
from astropy.io import fits
import os

filenames = np.loadtxt('basicredlist1.txt', dtype=str, ndmin=1)
basicredlist1_new = []
for f in filenames:
    h = fits.getheader(f)
    d = fits.getdata(f)
    print(f'fname:',f)
    print("NDIT header :", h.get("HIERARCH ESO DET NDIT"))
    print("Actual data shape :", d.shape)
    if h.get("HIERARCH ESO DET NDIT")<3:
        basicredlist1_new.append(f)

print(basicredlist1_new)
print(np.shape(filenames))
print(len(basicredlist1_new))

os.rename("basicredlist1.txt", "basicredlist1(copy).txt")
with open("basicredlist1.txt", "w") as f:
    for item in basicredlist1_new:
        f.write(f"{item}\n")

In [ ]:
irdis_make_asdi_cube('basicredlist1.txt', outfile='imcbasic1.save',szn=501)

sdat = np.load("imcbasic1.save.npz", allow_pickle=True)
cubeS = sdat["imc"].copy() 
img0=cubeS[0,:,:]
angs = sdat["pas"] # reads rotation angles list
print('total sky rotation (deg) =', zf.anglediff(angs[-1],angs[0]))
print('total sky rotation_sort (deg) =', zf.anglediff(np.sort(angs)[-1],np.sort(angs)[0]))
print(np.sort(angs))

In [ ]:
irdis_make_asdi_cube('basicredlist2.txt', outfile='imcbasic2.save',szn=501)

In [ ]:
#Save the original image cubes, before removing bad images.
import subprocess

subprocess.run("cp imcbasic1.save.npz imcbasic1.save.org.npz", shell=True)
subprocess.run("cp imcbasic2.save.npz imcbasic2.save.org.npz", shell=True)



In [ ]:
sph_remove_bad_images('imcbasic2.save.npz', inrad=8, outrad=40)
sph_remove_bad_images('imcbasic2.save.npz', inrad=40, outrad=200)
sph_remove_bad_images('imcbasic1.save.npz', inrad=8, outrad=40)
sph_remove_bad_images('imcbasic1.save.npz', inrad=40, outrad=200)

